# Sample inspector

Pick a model, a dataset, and a sample index. See the exact prompt the
eval pipeline sends — and what the model actually continues it with.

Two views below:

- **Single model** — full prompt + continuation for the `MODEL` you picked.
- **Three-model compare** — same prompt, side-by-side greedy continuations
  from exp_a / exp_b / exp_c so you can eyeball how they differ.


In [ ]:
from pathlib import Path
import textwrap

import torch
from datasets import load_dataset

from tiny_llm.eval._scoring import get_encoder
from tiny_llm.model.transformer import GPTModel
from tiny_llm.training.checkpoint import load_checkpoint
from tiny_llm.utils.device import resolve_device

device = resolve_device(None)
enc = get_encoder()

CKPT_DIR = Path("training_outputs")
if not CKPT_DIR.exists():
    CKPT_DIR = Path("../training_outputs")

_ds_cache: dict = {}
_model_cache: dict = {}

DATASET_SPECS = {
    "hellaswag":  ("Rowan/hellaswag", None, "validation"),
    "lambada":    ("EleutherAI/lambada_openai", "default", "test"),
    "winogrande": ("allenai/winogrande", "winogrande_xl", "validation"),
    "openbookqa": ("allenai/openbookqa", "main", "test"),
}


def get_dataset(name: str):
    if name not in _ds_cache:
        repo, cfg, split = DATASET_SPECS[name]
        kw = {"split": split, "trust_remote_code": False}
        _ds_cache[name] = load_dataset(repo, cfg, **kw) if cfg else load_dataset(repo, **kw)
    return _ds_cache[name]


def get_model(name: str):
    if name not in _model_cache:
        ckpt = str(CKPT_DIR / f"{name}_best.pt")
        meta = load_checkpoint(ckpt, model=None, device=device)
        m = GPTModel(meta["model_config"])
        load_checkpoint(ckpt, m, device=device)
        m.to(device).train(False)
        _model_cache[name] = m
        print(f"loaded {name}")
    return _model_cache[name]


def build_prompt(dataset: str, row: dict) -> tuple[str, str]:
    """Return (prompt_text, gold_answer_text) matching the benchmark format."""
    letters = "ABCDEFGHIJ"
    if dataset == "hellaswag":
        ctx = row["activity_label"] + " " + row["ctx_a"] + " " + row["ctx_b"]
        choices = row["endings"]
        gold = choices[int(row["label"])]
    elif dataset == "winogrande":
        ctx = row["sentence"].replace("_", "___")
        choices = [row["option1"], row["option2"]]
        gold = choices[int(row["answer"]) - 1]
    elif dataset == "openbookqa":
        ctx = "Question: " + row["question_stem"]
        choices = row["choices"]["text"]
        gold = choices[{"A": 0, "B": 1, "C": 2, "D": 3}[row["answerKey"]]]
    elif dataset == "lambada":
        words = row["text"].rsplit(" ", 1)
        return words[0], words[1]
    else:
        raise ValueError(dataset)
    parts = [ctx, ""]
    for i, c in enumerate(choices):
        parts.append(f"{letters[i]}) {c}")
    parts.append("")
    parts.append("Answer:")
    return "\n".join(parts), gold


def greedy_continuation(model, prompt_ids: list[int], n_gen: int) -> str:
    max_seq = model.context_length
    if len(prompt_ids) >= max_seq:
        prompt_ids = prompt_ids[-(max_seq - 1):]
    gen = list(prompt_ids)
    with torch.no_grad():
        for _ in range(n_gen):
            tail = gen[-max_seq:]
            out, _ = model(torch.tensor([tail], dtype=torch.long, device=device))
            gen.append(int(torch.argmax(out[0, -1, :]).item()))
    return enc.decode(gen[len(prompt_ids):])


print(f"Device: {device}")


In [ ]:
# ---- edit these and re-run the cells below ----
MODEL      = "exp_a"       # "exp_a" | "exp_b" | "exp_c"  (used by the single-model cell)
DATASET    = "winogrande"  # "hellaswag" | "winogrande" | "openbookqa" | "lambada"
SAMPLE_IDX = 0
N_GEN      = 40            # how many tokens to greedy-generate


In [ ]:
# Single model: full prompt and its greedy continuation.
ds = get_dataset(DATASET)
model = get_model(MODEL)
row = ds[SAMPLE_IDX]
prompt, gold = build_prompt(DATASET, row)
continuation = greedy_continuation(model, enc.encode(prompt), N_GEN)

print(f"{DATASET}[{SAMPLE_IDX}] / {len(ds)}   model={MODEL}   gold: {gold!r}")
print()
print("============================ PROMPT ============================")
print(prompt)
print("======================== END OF PROMPT =========================")
print(continuation)
print("============================== END =============================")


In [ ]:
# Three-model compare: same prompt, side-by-side greedy continuations.
ds = get_dataset(DATASET)
row = ds[SAMPLE_IDX]
prompt, gold = build_prompt(DATASET, row)
prompt_ids = enc.encode(prompt)

results = {}
for name in ("exp_a", "exp_b", "exp_c"):
    m = get_model(name)
    results[name] = greedy_continuation(m, prompt_ids, N_GEN)

print(f"{DATASET}[{SAMPLE_IDX}] / {len(ds)}   gold: {gold!r}")
print()
print("============================ PROMPT ============================")
print(prompt)
print("======================== END OF PROMPT =========================")
print()

col_width = 38
labels = list(results.keys())
# Collapse newlines into a visible separator so wrapping stays tidy.
wrapped = [
    textwrap.fill(results[n].replace("\n", " / "), width=col_width).split("\n")
    for n in labels
]
max_lines = max(len(w) for w in wrapped)
for w in wrapped:
    w.extend([""] * (max_lines - len(w)))

header = " | ".join(f"{lab:<{col_width}}" for lab in labels)
print(header)
print("-" * len(header))
for i in range(max_lines):
    print(" | ".join(f"{w[i]:<{col_width}}" for w in wrapped))
